# 🏦 Banking Customer EDA — Risk Analytics
> Exploratory Data Analysis on a 3,000-row banking customer dataset.

## 0. Imports & Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Create output folder for visuals
os.makedirs('visuals', exist_ok=True)

# Global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv('/content/Banking.csv')
print(f'Shape: {df.shape}')
df.head(5)

In [ ]:
df.info()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values detected.')

In [ ]:
# Descriptive statistics
df.describe()

## 2. Feature Engineering — Income Band

In [ ]:
bins = [0, 100_000, 300_000, float('inf')]
labels = ['Low', 'Medium', 'High']

df['Income Band'] = pd.cut(df['Estimated Income'], bins=bins, labels=labels, right=False)

print('Income Band distribution:')
print(df['Income Band'].value_counts())
print(f"\nAs % of total:")
print(df['Income Band'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = ['Low', 'Medium', 'High']
sns.countplot(data=df, x='Income Band', order=order, ax=ax)
ax.set_title('Income Band Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Income Band')
ax.set_ylabel('Number of Customers')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig('visuals/income_band.png')
plt.show()

## 3. Univariate Analysis — Categorical Features

In [ ]:
# Defined once and reused throughout
categorical_cols = [
    'BRId', 'GenderId', 'IAId', 'Amount of Credit Cards',
    'Nationality', 'Fee Structure', 'Loyalty Classification',
    'Properties Owned', 'Risk Weighting', 'Income Band'
]
# Note: 'Occupation' excluded from plots due to 195 unique categories

numerical_cols = [
    'Estimated Income', 'Superannuation Savings', 'Credit Card Balance',
    'Bank Loans', 'Bank Deposits', 'Checking Accounts',
    'Saving Accounts', 'Foreign Currency Account', 'Business Lending'
]

In [ ]:
# Value counts for all categorical columns
for col in categorical_cols:
    print(f"Value Counts for '{col}':")
    display(df[col].value_counts())
    print()

In [ ]:
n_cols = 2
n_rows = (len(categorical_cols) + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, ax=axes[i], order=df[col].value_counts().index)
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Univariate Analysis — Categorical Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('visuals/univariate_categorical.png', bbox_inches='tight')
plt.show()

In [ ]:
# Nationality breakdown only — save separately for README
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x='Nationality', order=df['Nationality'].value_counts().index, ax=ax)
ax.set_title('Customer Distribution by Nationality', fontsize=14, fontweight='bold')
ax.set_xlabel('Nationality')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('visuals/univariate_nationality.png')
plt.show()

## 4. Bivariate Analysis — Categorical Features × Nationality

In [ ]:
# Exclude 'Nationality' from bivariate since it's the hue variable
bivariate_cols = [col for col in categorical_cols if col != 'Nationality']

n_cols = 2
n_rows = (len(bivariate_cols) + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(bivariate_cols):
    sns.countplot(data=df, x=col, hue='Nationality', ax=axes[i],
                  order=df[col].value_counts().index)
    axes[i].set_title(f'{col} × Nationality', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(title='Nationality', fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Bivariate Analysis — Categorical Features × Nationality', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('visuals/bivariate_categorical_nationality.png', bbox_inches='tight')
plt.show()

In [ ]:
# Loyalty × Nationality — save separately for README
fig, ax = plt.subplots(figsize=(9, 5))
sns.countplot(data=df, x='Loyalty Classification', hue='Nationality',
              order=df['Loyalty Classification'].value_counts().index, ax=ax)
ax.set_title('Loyalty Classification × Nationality', fontsize=14, fontweight='bold')
ax.set_xlabel('Loyalty Classification')
ax.set_ylabel('Count')
ax.legend(title='Nationality')
plt.tight_layout()
plt.savefig('visuals/bivariate_loyalty_nationality.png')
plt.show()

## 5. Numerical Analysis — Distributions

In [ ]:
plt.figure(figsize=(15, 12))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1)
    sns.histplot(df[col], kde=True)
    plt.title(col, fontweight='bold')
    plt.xlabel('')
plt.suptitle('Distribution of Numerical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('visuals/numerical_histograms.png')
plt.show()

## 6. Bivariate Analysis — Income Band × Numerical Features

In [ ]:
# Average financial metrics per income band
income_band_summary = df.groupby('Income Band', observed=True)[numerical_cols].mean().round(0)
print('Average financial metrics by Income Band:')
display(income_band_summary)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.boxplot(data=df, x='Income Band', y=col, order=['Low', 'Medium', 'High'], ax=axes[i])
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Numerical Features by Income Band', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('visuals/boxplot_income_band.png')
plt.show()

## 7. Correlation Matrix

In [ ]:
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='crest',
    fmt='.2f',
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visuals/correlation_heatmap.png')
plt.show()

In [ ]:
# Top correlated pairs (excluding self-correlations)
corr_pairs = (
    correlation_matrix
    .where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
corr_pairs = corr_pairs.reindex(corr_pairs['Correlation'].abs().sort_values(ascending=False).index)
print('Top 10 correlated feature pairs:')
display(corr_pairs.head(10))

## 8. Key Insights

1. **Bank Deposits is highly correlated with Checking, Saving, and Foreign Currency accounts:** customers with high deposit balances tend to hold substantial funds across all account types, pointing to a wealth concentration effect. These clients represent lower lending risk.

2. **Income distribution is dominated by the Medium band (51%):** over half the customer base earns between $100k and $300k. This segment requires careful risk calibration as they are neither clearly low-risk nor high-risk borrowers.

3. **Risk Weighting is skewed toward low risk:** 70%+ of customers sit at Risk Weighting 1 or 2, but the tail (levels 4–5) represents ~160 high-risk profiles that warrant dedicated monitoring to minimise potential losses.

4. **Credit Card Balance shows weak correlation with Income:** credit usage appears driven more by spending behavior than earning capacity, reinforcing the need for behavioral data in credit risk models rather than income alone.

5. **Loyalty tier Jade dominates (44%):** combined with the prevalence of the High fee structure (49%), this suggests a customer base that is both loyal and financially engaged, reducing the probability of sudden default or churn.